# Observer la vérification du mot de passe — CW-Nano

Reprend **ton notebook fonctionnel** et ajoute une acquisition de consommation **sans voltage glitch**.

Le trigger D13 (si présent dans ce firmware) marque le **début de la comparaison**, pas nécessairement la réception UART.

**Condition physique :** pour interpréter la trace comme une mesure de consommation, il faut que l’entrée de mesure du CW-Nano soit réellement raccordée à un montage de mesure de courant approprié. Un simple câblage UART + trigger ne suffit pas.


## 0 — Préparation logicielle et paramètres

In [14]:
# Exécuter dans le même environnement Python que le package ChipWhisperer.
# Si nécessaire : %pip install chipwhisperer pandas matplotlib
import time, csv, pathlib
import chipwhisperer as cw
import pandas as pd
from IPython.display import display

BAUD = 9600
WRONG_PASSWORD = b"12345678\n"  # 8 caractères incorrects attendus; adapter si nécessaire
BOOT_WAIT_S = 0.2
READ_WAIT_S = 0.25
OUT = pathlib.Path("voltage_glitch_results.csv")

# Commencer sans glitch et sans brancher la sortie crowbar sur VCC.
ENABLE_ACTIVE_GLITCH = False
HARDWARE_VERIFIED = False  # True UNIQUEMENT après contrôle du câblage et mesure VCC
MAX_ACTIVE_ATTEMPTS = 20
print("Version ChipWhisperer:", getattr(cw, "__version__", "inconnue"))

Version ChipWhisperer: 6.0.0


## 1 — Connexion au CW-Nano et configuration UART

In [15]:
try:
    scope = cw.scope()
except OSError as exc:
    raise RuntimeError(
        "Le ChipWhisperer est déjà utilisé par un autre notebook ou processus. "
        "Ferme l'autre connexion, redémarre le noyau Jupyter, puis relance cette cellule."
    ) from exc

print("Scope:", scope)
# TIO4 écoute le front montant de D13 au moment de la vérification.
scope.io.tio4 = "high_z"

# API spécifique au CW-Nano : repeat=0 désactive les impulsions.
scope.glitch.repeat = 0
scope.glitch.ext_offset = 0

# L'UART du firmware n'utilise PAS le protocole SimpleSerial.
# Sur certaines versions, cw.target(scope) ouvre l'UART brut comme ci-dessous.
target = cw.target(scope, cw.targets.SimpleSerial2)
target.baud = BAUD
print("UART ouvert à", target.baud, "bauds; glitch OFF")

RuntimeError: Le ChipWhisperer est déjà utilisé par un autre notebook ou processus. Ferme l'autre connexion, redémarre le noyau Jupyter, puis relance cette cellule.

## 2 — Test UART sans glitch

In [ ]:
def read_uart(n=150, wait=READ_WAIT_S):
    time.sleep(wait)
    data = target.ser.read(n)
    if isinstance(data, str):
        return data.encode("latin-1")
    return bytes(data)

def send_password(password=WRONG_PASSWORD):
    # Ne pas utiliser target.simpleserial_write ici : le firmware attend du texte brut.
    target.ser.write(password)
    return read_uart()

print("Données déjà en attente:", read_uart())
for i in range(3):
    response = send_password()
    print(f"Tentative {i+1}: {response!r}")
print("Résultat attendu : message ACCESS DENIED (ou rejet équivalent).")

Données déjà en attente: b''
Tentative 1: b''
Tentative 2: b''
Tentative 3: b''
Résultat attendu : message ACCESS DENIED (ou rejet équivalent).


## 3 — Vérification du trigger sans glitch

Cette étape permet de confirmer que D13 (broche physique 19 de l'ATmega) est bien relié à TIO4 (broche 16 du CW-Nano). `scope.capture()` attend le trigger : un *timeout* suggère un problème de déclenchement, pas nécessairement un problème UART. Si la sortie de mesure ADC n'est pas raccordée, la trace n'aura pas de signification électrique ; seul le déclenchement est vérifié.

In [ ]:
scope.glitch.repeat = 0
scope.adc.samples = 1000
scope.arm()
target.ser.write(WRONG_PASSWORD)
timed_out = bool(scope.capture())  # True = timeout
trigger_response = read_uart()
print("Timeout trigger :", timed_out)
print("Réponse UART:", repr(trigger_response))
if timed_out:
    print("STOP : vérifier D13 -> TIO4, GND, front montant et timing du firmware.")
else:
    print("Trigger détecté. Le test de référence fonctionne.")
TRIGGER_OK = not timed_out

(ChipWhisperer Scope WARNING|File cwnano.py:711) Timeout in cwnano capture()


Timeout trigger : True
Réponse UART: b''
STOP : vérifier D13 -> TIO4, GND, front montant et timing du firmware.


## 4 — Capture de consommation synchronisée, sans glitch

Exécuter après avoir vérifié la connexion UART, le trigger et le chemin de mesure. L’acquisition couvre jusqu’à ~667 µs à 7,5 MS/s pour 5000 échantillons. Si les opérations intéressantes se produisent avant D13, elles ne seront pas visibles dans cette capture.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ne déclenche aucun glitch
scope.glitch.repeat = 0
scope.glitch.ext_offset = 0
scope.io.tio4 = "high_z"
scope.adc.samples = 5000

def capture_password_trace(password=WRONG_PASSWORD):
    # Lire/vider une éventuelle réponse résiduelle
    _ = read_uart(n=200, wait=0.05)
    scope.arm()
    target.ser.write(password)
    timed_out = bool(scope.capture())
    response = read_uart(n=200, wait=0.15)
    if timed_out:
        raise RuntimeError(
            "Timeout : aucun front montant n'a été reçu sur TIO4/GPIO4. "
            "Vérifie que le firmware met bien D13 à HIGH pendant la comparaison, "
            "que D13 est relié à TIO4, et que GND est commun. "
            "Aucune trace valide n'a été produite."
        )
    trace = np.asarray(scope.get_last_trace(), dtype=float)
    if trace.size == 0:
        raise RuntimeError("Capture vide")
    return trace, response

trace, response = capture_password_trace(b"12345678\n")

# CW-Nano : échantillonnage nominal à 7,5 MS/s ; à confirmer pour ton matériel
SAMPLE_RATE_HZ = 7_500_000
time_us = np.arange(len(trace)) / SAMPLE_RATE_HZ * 1e6
plt.figure(figsize=(13, 4))
plt.plot(time_us, trace, linewidth=0.8)
plt.xlabel("Temps après front montant de D13 (µs)")
plt.ylabel("Amplitude ADC (unités du CW-Nano)")
plt.title("Activité électrique pendant la vérification du mot de passe")
plt.grid(alpha=0.3)
plt.show()
print("Réponse UART :", repr(response))
print("Échantillons :", len(trace), "| Durée :", round(len(trace)/SAMPLE_RATE_HZ*1e6, 1), "µs")


(ChipWhisperer Scope WARNING|File cwnano.py:711) Timeout in cwnano capture()


RuntimeError: Timeout : aucun front montant n'a été reçu sur TIO4/GPIO4. Vérifie que le firmware met bien D13 à HIGH pendant la comparaison, que D13 est relié à TIO4, et que GND est commun. Aucune trace valide n'a été produite.

## 5 — Comparer plusieurs essais sans glitch

Une différence répétable entre les traces peut aider à distinguer les chemins d’exécution. Une différence isolée peut seulement provenir du bruit, d’un reset ou d’un décalage temporel.


In [16]:
passwords = [b"12345678\n", b"AAAAAAAA\n", b"ZZZZZZZZ\n"]
plt.figure(figsize=(13, 4))
for password in passwords:
    tr, reply = capture_password_trace(password)
    x = np.arange(len(tr)) / SAMPLE_RATE_HZ * 1e6
    plt.plot(x, tr, linewidth=0.75, alpha=0.8, label=f"{password!r}: {reply[:35]!r}")
plt.xlabel("Temps après trigger D13 (µs)")
plt.ylabel("Amplitude ADC")
plt.title("Comparaison de captures sans glitch")
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
plt.show()


(ChipWhisperer Scope WARNING|File cwnano.py:711) Timeout in cwnano capture()


RuntimeError: Timeout : aucun front montant n'a été reçu sur TIO4/GPIO4. Vérifie que le firmware met bien D13 à HIGH pendant la comparaison, que D13 est relié à TIO4, et que GND est commun. Aucune trace valide n'a été produite.

<Figure size 1300x400 with 0 Axes>

## 6 — Planned next steps (not completed)

**Current limitation:** The ChipWhisperer Nano did not detect the expected rising edge on TIO4. Consequently, the synchronized acquisition timed out. No usable power-consumption trace was obtained, and **no voltage glitch attack was performed or demonstrated**.

### Step 1 — Power analysis and identification of password verification

If the trigger issue had been resolved, the next step would have been to capture the ATmega328P's power consumption while sending an incorrect password over UART. By comparing several synchronized traces, we would have tried to identify the successive phases of execution, especially the moment when the program compares the received password with the stored password. This would have helped estimate the time window corresponding to the password-checking routine. Power traces alone would not prove exactly which instruction was executed.

### Step 2 — Voltage glitch during password verification

Once this time window had been estimated, the next planned step would have been to use the ChipWhisperer Nano to inject a short voltage disturbance during the password verification routine. We would then have monitored the UART response and the target's behaviour to distinguish a normal rejection from a reset, an execution fault, or a potential authentication bypass.

**Conclusion:** Because the rising edge could not be detected, these two steps remained theoretical. We cannot claim that password verification was located in a power trace or that authentication was bypassed.